In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, regexp_extract, to_date, col
from pyspark.sql.types import *

In [0]:
source_path = "s3://subscription-revenue-platform/raw/subscription_events/"

bronze_table_path = "s3://subscription-revenue-platform/bronze/subscription_events/"
bronze_checkpoint_path = "s3://subscription-revenue-platform/bronze/bronze_subscription_events_checkpoints/"
bronze_schema_path = "s3://subscription-revenue-platform/bronze/bronze_subscription_events_schema/"

In [0]:
catalog = "subscription_platform"
schema = "bronze"
bronze_table = f"{catalog}.{schema}.subscription_events"

In [0]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("customer_id", StringType()),
    StructField("subscription_id", StringType()),
    StructField("plan_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("currency", StringType()),
    StructField("event_time", TimestampType()),
    StructField("ingested_at", TimestampType()),
    StructField("source", StringType())
])

In [0]:
raw_events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", bronze_schema_path)
    .schema(event_schema)
    .load(source_path)
)

In [0]:
bronze_events = (
    raw_events
    .withColumn("processed_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
stream_query = (
    bronze_events.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

In [0]:
stream_query.awaitTermination()

In [0]:
%sql

SELECT *
FROM subscription_platform.bronze.subscription_events
ORDER BY processed_at DESC
LIMIT 20

event_id,event_type,customer_id,subscription_id,plan_id,amount,currency,event_time,ingested_at,source,processed_at,source_file
evt_1044d062a3514118bc6ff81eaaa9d677,SUBSCRIPTION_CANCELLED,cust_503,sub_1003,basic,1000.0,INR,2026-04-16T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-16/evt_1044d062a3514118bc6ff81eaaa9d677.json
evt_1db5e1d01dbb4e2aaba968a9e994a5e4,SUBSCRIPTION_REACTIVATED,cust_509,sub_1009,basic,1000.0,INR,2026-04-16T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-16/evt_1db5e1d01dbb4e2aaba968a9e994a5e4.json
evt_75ce558d7c164c59a992440fd26eaabe,SUBSCRIPTION_REACTIVATED,cust_502,sub_1002,pro,2000.0,INR,2026-04-16T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-16/evt_75ce558d7c164c59a992440fd26eaabe.json
evt_0b36ec83ac6445bda96e01f5820eb3dc,SUBSCRIPTION_REACTIVATED,cust_506,sub_1006,pro,2000.0,INR,2026-04-13T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-13/evt_0b36ec83ac6445bda96e01f5820eb3dc.json
evt_238d3a5c8aa9487ca395c0a8d6c8d87e,SUBSCRIPTION_REACTIVATED,cust_502,sub_1002,pro,2000.0,INR,2026-04-16T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-16/evt_238d3a5c8aa9487ca395c0a8d6c8d87e.json
evt_860446e8d3664ea7b1a64335ad0d33be,SUBSCRIPTION_REACTIVATED,cust_508,sub_1008,pro,2000.0,INR,2026-04-16T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-16/evt_860446e8d3664ea7b1a64335ad0d33be.json
evt_a8f7302cb286428599cbdd9734d9aa9c,SUBSCRIPTION_CREATED,cust_509,sub_1009,basic,1000.0,INR,2026-04-16T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-16/evt_a8f7302cb286428599cbdd9734d9aa9c.json
evt_94a7a2f1300948dcaaebc2e6c7c30157,SUBSCRIPTION_CANCELLED,cust_504,sub_1004,basic,1000.0,INR,2026-04-16T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-16/evt_94a7a2f1300948dcaaebc2e6c7c30157.json
evt_56f78b9c007441129cee733d17b9d846,SUBSCRIPTION_CANCELLED,cust_506,sub_1006,pro,2000.0,INR,2026-04-14T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-14/evt_56f78b9c007441129cee733d17b9d846.json
evt_f37740e1d2384db088b311b4ed6ffc54,SUBSCRIPTION_CREATED,cust_503,sub_1003,basic,1000.0,INR,2026-04-15T00:00:00.000Z,2026-04-16T00:00:00.000Z,billing_lambda,2026-04-28T19:04:51.722Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-15/evt_f37740e1d2384db088b311b4ed6ffc54.json


In [0]:
%sql

SELECT COUNT(*) AS total_records
FROM subscription_platform.bronze.subscription_events;

total_records
346


In [0]:
%sql

SELECT
    event_type,
    COUNT(*) AS event_count
FROM subscription_platform.bronze.subscription_events
GROUP BY event_type
ORDER BY event_count DESC;

event_type,event_count
SUBSCRIPTION_CREATED,90
SUBSCRIPTION_CANCELLED,88
SUBSCRIPTION_REACTIVATED,88
PLAN_CHANGED,80


In [0]:
%sql

SELECT
    MIN(event_time) AS earliest_event,
    MAX(event_time) AS latest_event
FROM subscription_platform.bronze.subscription_events;

earliest_event,latest_event
2026-03-29T00:00:00.000Z,2026-04-16T00:00:00.000Z


In [0]:
%sql

SELECT
    source_file,
    COUNT(*) AS records
FROM subscription_platform.bronze.subscription_events
GROUP BY source_file
ORDER BY records DESC;


source_file,records
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-06/evt_3ccf0b280f294dc58d106d60a4895232.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_3fbc4f4f38674c90901bce13521387ba.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-06/evt_4784afb5ffc540a9a5250fb9967f5221.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_12688d6cf7aa45be8d86c3d3173718e3.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_bd91aed92d4e483c9947f3fc4d0d0deb.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-05/evt_90e4426b0a8946168c28c3c49b967ec5.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_f53b4abc270c4f2098d45182463eb109.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_e51444a8693542a091cb9a40ce02f32a.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_49491cb9887c46658c66bf70a17bb7c9.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_d11ce11f411147bf84f8c41570a1ae36.json,1
